# Vakuutusyhtiön organisaatiohierarkian aiheuttaman korvauskäsittelyn vaihtelun purkaminen PROC NESTED -proseduurilla

## Tiivistelmä

Vahinkovakuutusyhtiö haluaa tietää, **mistä** korvauskäsittelyn epäjohdonmukaisuus
juontaa juurensa: johtuuko se maantieteellisten alueiden välisistä eroista,
konttoreiden välisistä eroista, yksittäisten vahinkotarkastajien eroista vai
pelkästä vahinkokohtaisesta kohinasta? Tämä muistikirja rakentaa synteettisen,
täysin sisäkkäisen autovahinkoaineiston (Alue > Konttori > Vahinkotarkastaja >
vahinko) ja käyttää **PROC NESTED** -proseduuria satunnaisvaikutusten
varianssianalyysiin, joka estimoi kunkin hierarkiatason varianssikomponentin ja
raportoi sen prosenttiosuutena kokonaisvaihtelusta. Tulos kertoo johdolle, mihin
organisaatiotasoon kannattaa kohdistaa toimet korvauskäsittelyn
yhdenmukaistamiseksi.

## Tietolähteet

Kaikki aineisto luodaan koodin sisällä ensimmäisessä DATA-vaiheessa — ei
ulkoisia tiedostoja, ei verkkoyhteyttä. Rakenne on **täysin sisäkkäinen**:
jokainen konttori kuuluu täsmälleen yhteen alueeseen, jokainen vahinkotarkastaja
täsmälleen yhteen konttoriin ja jokainen vahinko täsmälleen yhdelle
vahinkotarkastajalle.

| Tietojoukko | Rivit | Muuttuja | Tyyppi | Rooli | Kuvaus |
|---------|------|----------|------|------|-------------|
| `claims` | 40 | `Region` | merkkijono | LUOKKA (taso 1, uloin) | Maantieteellinen alue (5 aluetta: KO, KA, KS, LR, LO) |
| | | `Branch` | merkkijono | LUOKKA (taso 2, alueen sisällä) | Konttorin tunnus (2 konttoria per alue) |
| | | `Adjuster` | merkkijono | LUOKKA (taso 3, konttorin sisällä) | Vahinkotarkastajan tunnus (2 tarkastajaa per konttori) |
| | | `Settlement` | numero | VASTEMUUTTUJA | Autovahingosta maksettu korvaus, USD |
| | | `CycleDays` | numero | VASTEMUUTTUJA | Päivät vahinkoilmoituksesta korvauksen maksuun |

Synteettinen rakenne: 5 aluetta x 2 konttoria x 2 tarkastajaa x 2 vahinkoa = 40
havaintoa. Aluetason satunnaisvaikutus, konttori-alueen-sisällä-satunnaisvaikutus,
tarkastaja-konttorin-sisällä-satunnaisvaikutus ja vahinkotason jäännöstermi
kerrostetaan additiivisesti `rand('NORMAL', ...)`-funktiolla, jotta
varianssikomponentit ovat palautettavissa. Aluevaikutus piirretään suurimmalla
keskihajonnalla (2 200), jotta *missä* vahinko käsitellään on hallitseva tekijä.
Siemenluku kiinnitetty komennolla `call streaminit(20260531)`. Kompakti, täysin
tasapainoinen asetelma pitää jokaisen hierarkiatason täytettynä todellisilla
vapausastein.

# Korvauskäsittelyn vaihtelun purkaminen PROC NESTED -proseduurilla

Kun vahinkovakuutusyhtiö tarkastelee, kuinka paljon se maksaa autovahinkojen
korvauksissa, se havaitsee usein suurta vaihtelua. Osa vaihtelusta on
väistämätöntä (jokainen onnettomuus on erilainen), mutta osa heijastaa
**organisatorisia** tekijöitä — yksi alue on toista anteliaampi, jokin konttori
maksaa muita anteliaammin, tai yksittäinen vahinkotarkastaja erottuu poikkeavana.

Aineistolla on tiukasti **sisäkkäinen** (hierarkkinen) rakenne:

```
Alue  ->  Konttori (alueen sisällä)  ->  Vahinkotarkastaja (konttorin sisällä)  ->  yksittäiset vahingot
```

Konttori kuuluu täsmälleen yhteen alueeseen ja vahinkotarkastaja täsmälleen
yhteen konttoriin, joten tekijät ovat sisäkkäisiä eivätkä risteäviä.
`PROC NESTED` suorittaa satunnaisvaikutusten varianssianalyysin juuri
tällaiselle asetelmalle ja estimoi **varianssikomponentin** jokaisella tasolla,
ilmaistuna prosenttiosuutena kokonaisvaihtelusta. Tämä prosenttijakauma vastaa
liiketoiminnalliseen kysymykseen: *mihin organisaation tasoon kannattaa
kohdistaa toimet, jotta korvaukset olisivat yhdenmukaisempia?*

## Vaihe 1 — Luo synteettinen, täysin sisäkkäinen vahinkoaineisto

Simuloimme 5 aluetta, joista jokaisessa on 2 konttoria, joista jokaisessa on 2
vahinkotarkastajaa, joista jokainen käsittelee 2 vahinkoa (40 riviä yhteensä).
Rakennamme vasteen additiivisista satunnaisvaikutuksista, jotta jokainen taso
todella tuo oman varianssinsa:

- **alue**vaikutus (suurin hajonta — alueet eroavat eniten),
- **konttori-alueen-sisällä** -vaikutus,
- **tarkastaja-konttorin-sisällä** -vaikutus,
- ja **vahinkotason jäännöstermi** (tarkastajan sisäinen kohina).

`call streaminit` kiinnittää siemenluvun toistettavuutta varten; jokainen
vaikutus piirretään funktiolla `rand('NORMAL', mean, sd)`. Aluevaikutuksella on
laajin keskihajonta, joten sen pitäisi viedä suurin osuus kokonaisvarianssista.
Luomme myös toisen vastemuuttujan, `CycleDays`, joka jakaa saman hierarkian,
jotta voimme myöhemmin havainnollistaa moni-vastemuuttuja-analyysin.

In [1]:
TIEDOT claims;
   CALL streaminit(20260531);
   PITUUS Region $2 Branch $6 Adjuster $9;

   /* Region-level random effect: regions differ the most. */
   TEE r = 1 ASTI 5;
      Region = SCAN('KO KA KS LR LO', r);
      RegionEffect  = rand('NORMAL', 0, 2200);
      RegionCycle   = rand('NORMAL', 0, 11);

      /* Branch nested within region. */
      TEE b = 1 ASTI 2;
         Branch = catx('-', Region, KIRJOITA(b, z2.));
         BranchEffect = rand('NORMAL', 0, 700);
         BranchCycle  = rand('NORMAL', 0, 4);

         /* Adjuster nested within branch. */
         TEE a = 1 ASTI 2;
            Adjuster = catx('-', Branch, KIRJOITA(a, z1.));
            AdjEffect = rand('NORMAL', 0, 450);
            AdjCycle  = rand('NORMAL', 0, 2.5);

            /* Individual claims handled by this adjuster. */
            TEE claim = 1 ASTI 2;
               Settlement = 8500
                          + RegionEffect + BranchEffect + AdjEffect
                          + rand('NORMAL', 0, 1100);
               CycleDays  = 21
                          + RegionCycle + BranchCycle + AdjCycle
                          + rand('NORMAL', 0, 6);
               JOS Settlement < 0 NIIN Settlement = 0;
               JOS CycleDays  < 1 NIIN CycleDays  = 1;
               TULOSTE;
            LOPPU;
         LOPPU;
      LOPPU;
   LOPPU;

   SÄILYTÄ Region Branch Adjuster Settlement CycleDays;
   NIMIKE Region   = "Alue"
         Branch   = "Konttori"
         Adjuster = "Vahinkotarkastaja";
SUORITA;


NOTE: DATA claims


NOTE: Wrote claims (40 rows, 5 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds


## Vaihe 2 — Lajittele sisäkkäisyyshierarkian mukaan

`PROC NESTED` vaatii, että syöttöaineisto on **lajiteltu LUOKKA-muuttujien
mukaan siinä järjestyksessä, jossa ne luetellaan** — uloin tekijä ensin.
Lajittelemme muuttujien `Region Branch Adjuster` mukaan, jotta proseduuri voi
kulkea hierarkian läpi oikein.

In [2]:
PROSEDUURI LAJITTELE TIEDOT=claims;
   MUKAAN Region Branch Adjuster;
SUORITA;


NOTE: PROC SORT data=claims

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: Read 40 rows from claims.
NOTE: Wrote claims (40 rows, 5 columns).
NOTE: PROC SORT statement used.


## Vaihe 3 — Korvaussumman varianssikomponenttianalyysi

Ydinanalyysi. Luettelemme LUOKKA-muuttujat uloimmasta sisimpään (`Region
Branch Adjuster`); sisin toisto — yksittäiset vahingot — käsitellään
automaattisesti virhetermina. `VAR Settlement` nimeää yhden vastemuuttujan.

`LABEL`-lause antaa vasteelle ystävällisemmän näyttönimen tulostebannerissa.
Tuloste tuottaa odotettujen neliösummien kertoimet, varianssianalyysitaulukon
(jossa jokaisen varianssikomponentin F-testi nollaa vastaan) sekä
**varianssikomponentti**-estimaatin jokaisella tasolla yhdessä sen
**prosenttiosuuden kokonaisuudesta** kanssa.

In [3]:
OTSIKKO 'Autovahinkojen korvausten varianssikomponentit';
title2 'Alue / Konttori / Tarkastaja: satunnaisvaikutusten ANOVA';

PROSEDUURI nested TIEDOT=claims;
   LUOKKA Region Branch Adjuster;
   MUUTTUJA Settlement;
   NIMIKE Settlement = 'Maksettu korvaus (USD)';
SUORITA;

                                     Autovahinkojen korvausten varianssikomponentit                                     
                                Alue / Konttori / Tarkastaja: satunnaisvaikutusten ANOVA                                


                       Nested Random Effects Analysis of Variance

Nested Random Effects Analysis of Variance for Variable Maksettu korvaus (USD)
Variance Source       DF  Sum of Squares       F Value      Pr > F  Error Term   Mean Square  Variance Component    Percent of Total    EMS Coef
Alue                   4  62114122.126866          6.71      0.0304  Konttori  15528530.531717  1651824.844989             54.4057      8.0000
Konttori               5  11569658.859028          1.89      0.1829  Vahinkotarkastaja  2313931.771806   272682.828942              8.9813      4.0000
Vahinkotarkastaja     10  12232004.560376          1.22      0.3348     Error  1223200.456038   111582.782346              3.6752      2.0000
Error                 20  200


NOTE: Option TITLE changed to Autovahinkojen korvausten varianssikomponentit.
NOTE: Option TITLE2 changed to Alue / Konttori / Tarkastaja: satunnaisvaikutusten ANOVA.
NOTE: PROC NESTED nested ANOVA / variance components

NOTE: PROC NESTED statement used.


## Vaihe 4 — Analysoi kaksi vastetta samanaikaisesti

Johto on kiinnostunut myös **käsittelyajasta** — kuinka kauan vahinkojen
käsittely kestää. Lisäämme `CycleDays`-muuttujan `VAR`-listaan. Kun
vastemuuttujia on useampi kuin yksi, `PROC NESTED` raportoi lisäksi
**kovarianssianalyysin**, joka näyttää, miten kaksi vastetta vaihtelevat
yhdessä hierarkian jokaisella tasolla (esimerkiksi, maksavatko samat alueet
enemmän myös hitaammin).

In [4]:
OTSIKKO 'Korvaussumman ja käsittelyajan varianssikomponentit';
title2 'Kaksi vastemuuttujaa samassa sisäkkäisessä hierarkiassa';

PROSEDUURI nested TIEDOT=claims;
   LUOKKA Region Branch Adjuster;
   MUUTTUJA Settlement CycleDays;
   NIMIKE Settlement = 'Maksettu korvaus (USD)'
         CycleDays  = 'Käsittelyaika (pv)';
SUORITA;

                                  Korvaussumman ja käsittelyajan varianssikomponentit                                   
                                Kaksi vastemuuttujaa samassa sisäkkäisessä hierarkiassa                                 


                       Nested Random Effects Analysis of Variance

Nested Random Effects Analysis of Variance for Variable Maksettu korvaus (USD)
Variance Source       DF  Sum of Squares       F Value      Pr > F  Error Term   Mean Square  Variance Component    Percent of Total    EMS Coef
Alue                   4  62114122.126866          6.71      0.0304  Konttori  15528530.531717  1651824.844989             54.4057      8.0000
Konttori               5  11569658.859028          1.89      0.1829  Vahinkotarkastaja  2313931.771806   272682.828942              8.9813      4.0000
Vahinkotarkastaja     10  12232004.560376          1.22      0.3348     Error  1223200.456038   111582.782346              3.6752      2.0000
Error                 20  200


NOTE: Option TITLE changed to Korvaussumman ja käsittelyajan varianssikomponentit.
NOTE: Option TITLE2 changed to Kaksi vastemuuttujaa samassa sisäkkäisessä hierarkiassa.
NOTE: PROC NESTED nested ANOVA / variance components

NOTE: PROC NESTED statement used.


## Vaihe 5 — Pelkkä varianssinäkymä AOV-optiolla

Nopeaa varianssikomponenttiyhteenvetoa varten molemmille vasteille `AOV`-optio
rajaa tulosteen varianssianalyysitilastoihin ja **jättää pois
kovarianssianalyysin**. Tämä on tiivis näkymä, jonka salkun analyytikko
silmäilisi, kun hän tarvitsee vain kunkin vasteen tasokohtaisen
varianssijakauman, ei vasteiden välistä kovarianssia.

In [5]:
OTSIKKO 'Vain varianssikomponentit (AOV)';

PROSEDUURI nested TIEDOT=claims aov;
   LUOKKA Region Branch Adjuster;
   MUUTTUJA Settlement CycleDays;
   NIMIKE Settlement = 'Maksettu korvaus (USD)'
         CycleDays  = 'Käsittelyaika (pv)';
SUORITA;

OTSIKKO;

                                            Vain varianssikomponentit (AOV)                                             
                                Kaksi vastemuuttujaa samassa sisäkkäisessä hierarkiassa                                 


                       Nested Random Effects Analysis of Variance

Nested Random Effects Analysis of Variance for Variable Maksettu korvaus (USD)
Variance Source       DF  Sum of Squares       F Value      Pr > F  Error Term   Mean Square  Variance Component    Percent of Total    EMS Coef
Alue                   4  62114122.126866          6.71      0.0304  Konttori  15528530.531717  1651824.844989             54.4057      8.0000
Konttori               5  11569658.859028          1.89      0.1829  Vahinkotarkastaja  2313931.771806   272682.828942              8.9813      4.0000
Vahinkotarkastaja     10  12232004.560376          1.22      0.3348     Error  1223200.456038   111582.782346              3.6752      2.0000
Error                 20  200


NOTE: Option TITLE changed to Vain varianssikomponentit (AOV).
NOTE: PROC NESTED nested ANOVA / variance components

NOTE: PROC NESTED statement used.


## Tulosten tulkinta

Varianssianalyysitaulukon **Percent of Total** -sarake on pääkohta. Sen
lukeminen ylhäältä alas kertoo, kuinka suuri osuus korvaussumman
kokonaisvaihtelusta on peräisin kustakin organisaatiotasosta. Korvaussumman
ajolla tulos on:

- **Alue — 54,4 %** (Pr > F = 0,0304). Aineisto luotiin suurimmalla
  aluetason hajonnalla, ja analyysi löytää sen: yli puolet kaikesta
  korvaussumman vaihtelusta on alueiden *välistä*, tilastollisesti merkitsevä
  näyttö siitä, että *missä* vahinko käsitellään on hallitseva tekijä.
- **Konttori (alueen sisällä) — 9,0 %** (Pr > F = 0,18). Vaatimaton lisäosuus,
  kun porataan alueesta yksittäisiin konttoreihin; ei tilastollisesti
  merkitsevä tässä.
- **Vahinkotarkastaja (konttorin sisällä) — 3,7 %** (Pr > F = 0,33). Vain
  vähän vaihtelua on jäljitettävissä yksittäiseen vahinkotarkastajaan tässä
  vahinkosalkussa.
- **Virhe — 32,9 %.** Jäännöksenä jäävä vahinkokohtainen kohina tarkastajan
  sisällä; tämä on jäännösvaihtelua, jota mikään organisatorinen vipu ei voi
  poistaa.

Jokaisella tasolla on myös **F-testi (Pr > F)** nollahypoteesille, jonka
mukaan sen varianssikomponentti on nolla. Alueen pieni p-arvo (0,0304) on
tilastollista näyttöä todellisista systemaattisista eroista alueiden välillä,
ei otantasattumasta.

Käsittelyaikaa koskeva vaste kertoo rinnakkaisen tarinan: **Alue selittää
45,8 %** käsittelypäivien vaihtelusta (Pr > F = 0,0448, jälleen merkitsevä),
Konttorin ja Vahinkotarkastajan osuuksien jäädessä yksinumeroisiksi ja
jäännöksen kantaessa 40,1 %. Sekä *kuinka paljon* vakuutusyhtiö maksaa että
*kuinka kauan* se kestää määräytyvät siis ensisijaisesti alueen mukaan.

Kaksivastelaji tuottaa lisäksi **kovarianssianalyysin**. Tässä Alue-taso
ohjaa ristitulojen vaihtelua, ja kokonaiskorrelaatiokerroin on **-0,36** —
negatiivinen yhteys: alueet, jotka maksavat suurempia korvauksia,
käsittelevät ne yleensä *nopeammin*, ei hitaammin. Tämä on hyödyllinen,
ei-ilmeinen havainto — kalliit alueet eivät ole hitaita.

**Liiketoiminnallinen johtopäätös:** koska Alue hallitsee prosenttiosuusjakaumaa
molemmissa vasteissa, vakuutusyhtiön kannattaa yhdenmukaistaa korvausohjeet ja
valtuusrajat *alueiden välillä* ensin — siellä on suurin, tilastollisesti
merkitsevä epäjohdonmukaisuus — ennen kuin investoidaan konttori- tai
tarkastajatason toimenpiteisiin. Korvaussumman ja käsittelyajan negatiivinen
korrelaatio tarkoittaa, ettei yhtään aluetta ole sekä kallein että hitain,
joten nämä kaksi ongelmaa voidaan ratkaista erillisillä, aluekohtaisilla
ohjelmilla. PROC NESTED muuttaa epämääräisen tunteen "korvauksemme ovat
epäjohdonmukaisia" toimintakelpoiseksi, taso tasolta eteneväksi selvitykseksi
siitä, mistä epäjohdonmukaisuus juontaa juurensa.